In [ ]:
import pandas as pd
import numpy as np

import json
with open('config.json', 'r') as f:
    config = json.load(f)
    
from helpers.sementic_recoding import drop_by_threshold,recode_semantic_missingness
from helpers.modeling import (
    prepare_data,
    identify_column_types,
    create_preprocessor,
    evaluate_model,
    run_grid_search,
)
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR, NuSVR  # regression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv(config['filepath'], encoding="latin-1", header=[0, 1])
print(f"Initial shape: {df.shape}")

Initial shape: (2189, 92)


In [2]:
df.columns = df.columns.get_level_values(1)

column_names = config['column_names']
df.columns = column_names

df = df.iloc[1:].reset_index(drop=True)
df = df.drop(columns=['mix_id', 'country', 'reference_details', 'publish_year'])

for col in df.columns:
    converted = pd.to_numeric(df[col], errors='coerce')
    if converted.isna().sum() == df[col].isna().sum():
        df[col] = converted

print(f"Cleaned shape: {df.shape}")
print(f"Total columns: {len(df.columns)}")
df['paper_reference'] = df['paper_reference'].ffill()
df.head()

Cleaned shape: (2188, 88)
Total columns: 88


,cement,cement_type,cement_grade,silica_fume,fly_ash,fly_ash_type,limestone_powder,quartz_powder,glass_powder,rice_husk_ash,...,shrinkage_standard,shrinkage_28d,shrinkage_56d,freeze_thaw_standard,freeze_thaw_cycles,rcpt_standard,rcpt,resistivity_standard,surface_resistivity,paper_reference
0,839.0,Type I/II low-alkali portland cement,NaN,104.0,104.0,Class-F,0.0,0.0,0.0,0.0,...,ASTM C157,270.0,371.0,ASTM C666,105.0,ASTM C1202,165.0,AASHTO T 358,386.0,Ref-1-data
1,839.0,Type I/II low-alkali portland cement,NaN,104.0,52.0,Class-F,0.0,0.0,0.0,0.0,...,ASTM C157,242.0,317.0,ASTM C666,106.0,ASTM C1202,142.0,AASHTO T 358,424.0,Ref-1-data
2,839.0,Type I/II low-alkali portland cement,NaN,104.0,26.0,Class-F,0.0,0.0,0.0,0.0,...,ASTM C157,223.0,290.0,ASTM C666,106.0,ASTM C1202,136.0,AASHTO T 358,463.0,Ref-1-data
3,839.0,Type I/II low-alkali portland cement,NaN,104.0,0.0,NaN,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ref-1-data
4,839.0,Type I/II low-alkali portland cement,NaN,104.0,52.0,Class-F,0.0,0.0,0.0,0.0,...,ASTM C157,231.0,286.0,ASTM C666,106.0,ASTM C1202,135.0,AASHTO T 358,437.0,Ref-1-data


In [3]:
# Remove rows with missing target variable
df = df.dropna(subset=['cs_28d'])
print(f"Rows with 28-day compressive strength data: {len(df)}")

# Define columns to drop (focusing on cs_28d as target)
drop_cols = config["drop_cols"]

# Drop unnecessary columns
df = df.drop(columns=drop_cols)
print(f"Final shape after filtering: {df.shape}")
print(f"Remaining columns: {df.shape[1]}")



Rows with 28-day compressive strength data: 2073
Final shape after filtering: (2073, 44)
Remaining columns: 44


In [4]:
df_cleaned = pd.read_csv(config["initial_cleaned_filepath"], index_col=0)
df_cleaned['paper_reference'] = df['paper_reference'].values
df_cleaned.iloc[1]
print(f"Final shape after filtering: {df.shape}")
print(f"Remaining columns: {df.shape[1]}")
df_cleaned.head(20)

Final shape after filtering: (2073, 44)
Remaining columns: 44


,cement,cement_type,cement_grade,silica_fume,fly_ash,fly_ash_type,limestone_powder,quartz_powder,glass_powder,rice_husk_ash,...,fiber2_youngs_modulus,water,sp_type,sp_amount,curing_method,curing_temp,curing_humidity,curing_pressure,cs_28d,paper_reference
0,839.0,OPC_I,NaN,104.0,104.0,class F,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,56.50,Standard Curing,NaN,NaN,NaN,135.0,Ref-1-data
1,839.0,OPC_I,NaN,104.0,52.0,class F,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,59.33,Standard Curing,NaN,NaN,NaN,132.0,Ref-1-data
2,839.0,OPC_I,NaN,104.0,26.0,class F,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,59.33,Standard Curing,NaN,NaN,NaN,122.5,Ref-1-data
3,839.0,OPC_I,NaN,104.0,0.0,NaN,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,62.15,Standard Curing,NaN,NaN,NaN,116.0,Ref-1-data
4,839.0,OPC_I,NaN,104.0,52.0,class F,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,64.98,Standard Curing,NaN,NaN,NaN,134.0,Ref-1-data
5,839.0,OPC_I,NaN,104.0,26.0,class F,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,64.98,Standard Curing,NaN,NaN,NaN,131.5,Ref-1-data
6,839.0,OPC_I,NaN,104.0,0.0,NaN,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,67.80,Standard Curing,NaN,NaN,NaN,130.5,Ref-1-data
7,839.0,OPC_I,NaN,104.0,83.0,class F,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,59.33,Standard Curing,NaN,NaN,NaN,134.5,Ref-1-data
8,839.0,OPC_I,NaN,104.0,62.0,class F,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,62.15,Standard Curing,NaN,NaN,NaN,132.5,Ref-1-data
9,839.0,OPC_I,NaN,104.0,0.0,NaN,0.0,0.0,0.0,0.0,...,NaN,147.0,PCE_SP,64.98,Standard Curing,NaN,NaN,NaN,123.0,Ref-1-data


In [5]:
## Semantic Missingness Recoding

# Define material amount/type pairs for semantic analysis
amount_type_twins = config["amount_type_twins"]

# Apply semantic recoding
df_semantic_recoded = df_cleaned.copy()

for amount_col, type_col in amount_type_twins:
    before_amount = df[amount_col].isna().sum()
    before_type = df[type_col].isna().sum()
    
    recode_semantic_missingness(df_semantic_recoded, amount_col, type_col)
    
    print(f"\n{amount_col} / {type_col}")
    print(f"  NaN amount:      {before_amount} → {df[amount_col].isna().sum()}  (true reporting gap)")
    print(f"  NaN type:        {before_type} → {df[type_col].isna().sum()}  (true reporting gap)")
    print(f"  unknown_type:    {(df[type_col] == 'unknown_type').sum()}  (recoded)")
    print(f"  not_applicable:  {(df[type_col] == 'not_applicable').sum()}  (recoded)")
    
# Recoded + 50% missing data threshold
df_semantic_recod_50 = df_semantic_recoded.copy()
df_semantic_recod_50 = drop_by_threshold(df_semantic_recod_50, 0.5)
print(f"Recoded + 50% threshold shape: {df_semantic_recod_50.shape}")


fly_ash / fly_ash_type
  NaN amount:      0 → 0  (true reporting gap)
  NaN type:        1647 → 1647  (true reporting gap)
  unknown_type:    0  (recoded)
  not_applicable:  0  (recoded)

slag / slag_type
  NaN amount:      0 → 0  (true reporting gap)
  NaN type:        1981 → 1981  (true reporting gap)
  unknown_type:    0  (recoded)
  not_applicable:  0  (recoded)

filler / filler_type
  NaN amount:      0 → 0  (true reporting gap)
  NaN type:        1713 → 1713  (true reporting gap)
  unknown_type:    0  (recoded)
  not_applicable:  0  (recoded)

sand / sand_type
  NaN amount:      0 → 0  (true reporting gap)
  NaN type:        82 → 82  (true reporting gap)
  unknown_type:    0  (recoded)
  not_applicable:  0  (recoded)

fiber1_amount / fiber1_type
  NaN amount:      669 → 669  (true reporting gap)
  NaN type:        669 → 669  (true reporting gap)
  unknown_type:    0  (recoded)
  not_applicable:  0  (recoded)

fiber2_amount / fiber2_type
  NaN amount:      1952 → 1952  (true repo

In [6]:
df_semantic_recod_50 = df_semantic_recod_50.drop(columns=['cement_grade']) # cement grade 38% missing, encodes sames info as cement type
fiber_cols = ['fiber1_length', 'fiber1_diameter']  
df_semantic_recod_50[fiber_cols] = df_semantic_recod_50[fiber_cols].fillna(0)  # replacing with mean wrong
df_semantic_recod_50.head()

df_semantic_recod_50.to_csv("../Datasets/processed/UHPC_dataset/semantic_recoding_features_50_with_publications.csv", index=False)

In [13]:
from sklearn.model_selection import GroupShuffleSplit

df = df_semantic_recod_50
target_col = 'cs_28d'
group_col  = 'paper_reference'  

X = df.drop(columns=[target_col, group_col])
y = df[target_col]  
groups = df[group_col]


# 1. carve out test
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_val_idx, test_idx = next(gss.split(X, y, groups=groups))

# 2. carve val from remainder
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.15/0.85, random_state=42)
train_idx_rel, val_idx_rel = next(gss2.split(
    X.iloc[train_val_idx], y.iloc[train_val_idx], groups=groups.iloc[train_val_idx]
))

X_train = X.iloc[train_val_idx].iloc[train_idx_rel]
X_val   = X.iloc[train_val_idx].iloc[val_idx_rel]
X_test  = X.iloc[test_idx]
y_train = y.iloc[train_val_idx].iloc[train_idx_rel]
y_val   = y.iloc[train_val_idx].iloc[val_idx_rel]
y_test  = y.iloc[test_idx]

print(f"Train set size: {X_train.shape}")
print(f"Validation set size: {X_val.shape}")
print(f"Test set size: {X_test.shape}")

numerical_cols, one_hot_columns, k_fold_columns = identify_column_types(X)
preprocessor = create_preprocessor(numerical_cols, one_hot_columns, k_fold_columns,
                                   handle_unknown='ignore')

print(f"After encoding Training Set Shape: {preprocessor.fit_transform(X_train, y_train).shape}")
print(f"After encoding Validation Set Shape: {preprocessor.transform(X_val).shape}")
print(f"After encoding Test Set Shape: {preprocessor.transform(X_test).shape}")

Train set size: (1461, 33)
Validation set size: (329, 33)
Test set size: (283, 33)
After encoding Training Set Shape: (1461, 60)
After encoding Validation Set Shape: (329, 60)
After encoding Test Set Shape: (283, 60)


In [14]:
# KNN PIPELINE CREATION
knn_pipeline = Pipeline([('preprocessor', preprocessor), ('model', KNeighborsRegressor())])

knn_grid = {
    'model__n_neighbors': list(range(1, 31)),
    'model__weights': ['uniform', 'distance'],
    'model__p': [1, 2, 3],
    'model__metric': ['euclidean', 'manhattan', 'minkowski']
}

gs_knn = run_grid_search(knn_pipeline, knn_grid, X_train, X_val, X_test, y_train, y_val, y_test, 'KNN')


GridSearchCV will test 540 combinations for KNN...

BEST HYPERPARAMETERS — KNN
  metric: manhattan
  n_neighbors: 12
  p: 2
  weights: distance
Best CV RMSE: 26.4310


c:\Users\shoai\anaconda3\envs\aicome\Lib\site-packages\threadpoolctl.py:1214: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)



VALIDATION SET PERFORMANCE
RMSE: 39.0417
MAE: 28.5299
R2: 0.3696
Correlation: 0.7572
Mean_Residual: 18.2249
N: 329

TEST SET PERFORMANCE
RMSE: 20.1772
MAE: 15.4321
R2: 0.5484
Correlation: 0.7788
Mean_Residual: -7.2092
N: 283

RESULTS SUMMARY
          Metric  Validation       Test
           RMSE   39.041720  20.177239
            MAE   28.529911  15.432070
             R2    0.369631   0.548392
Correlation (R)    0.757183   0.778760
  Mean Residual   18.224854  -7.209229
              N  329.000000 283.000000

TOP 10 COMBINATIONS
   metric  n_neighbors  p  weights  mean_test_score   CV_rmse
manhattan           12  2 distance      -698.597548 26.430996
manhattan           11  2 distance      -701.256210 26.481243
minkowski           13  1 distance      -701.906248 26.493513
manhattan           11  1 distance      -701.967626 26.494672
manhattan           12  1 distance      -702.530144 26.505285
minkowski           11  1 distance      -703.568363 26.524863
manhattan           11  3 di

In [15]:
# SVR PIPELINE CREATION
svr_pipeline = Pipeline([('preprocessor', preprocessor), ('model', SVR(kernel='rbf'))])

svr_grid = {
    'model__C':        [128, 256, 512, 1024],
    'model__epsilon': [0.01, 0.1, 0.5, 1, 3]
}

gs_svr = run_grid_search(svr_pipeline, svr_grid, X_train, X_val, X_test, y_train, y_val, y_test, 'SVR')


GridSearchCV will test 20 combinations for SVR...

BEST HYPERPARAMETERS — SVR
  C: 128
  epsilon: 0.01
Best CV RMSE: 30.2895

VALIDATION SET PERFORMANCE
RMSE: 43.3641
MAE: 32.7868
R2: 0.2223
Correlation: 0.6656
Mean_Residual: 20.5570
N: 329

TEST SET PERFORMANCE
RMSE: 22.9165
MAE: 17.4468
R2: 0.4174
Correlation: 0.6937
Mean_Residual: -5.5504
N: 283

RESULTS SUMMARY
          Metric  Validation       Test
           RMSE   43.364068  22.916498
            MAE   32.786787  17.446766
             R2    0.222327   0.417448
Correlation (R)    0.665641   0.693711
  Mean Residual   20.557000  -5.550442
              N  329.000000 283.000000

TOP 10 COMBINATIONS
  C  epsilon  mean_test_score   CV_rmse
128     0.01      -917.456212 30.289540
128     1.00      -953.847833 30.884427
128     3.00      -955.022547 30.903439
128     0.10      -955.476378 30.910781
128     0.50      -966.520171 31.088908
256     3.00      -973.884230 31.207118
256     0.50      -995.231444 31.547289
256     0.10    

In [16]:
# NuSVR PIPELINE CREATION
nusvr_pipeline = Pipeline([('preprocessor', preprocessor), ('model', NuSVR(kernel='rbf'))])

nusvr_grid = {
    'model__C':   [128, 256, 512, 1024],
    'model__nu': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
}

gs_nusvr = run_grid_search(nusvr_pipeline, nusvr_grid, X_train, X_val, X_test, y_train, y_val, y_test, 'NuSVR')


GridSearchCV will test 36 combinations for NuSVR...

BEST HYPERPARAMETERS — NuSVR
  C: 128
  nu: 0.4
Best CV RMSE: 30.2990

VALIDATION SET PERFORMANCE
RMSE: 40.8252
MAE: 30.6253
R2: 0.3107
Correlation: 0.7329
Mean_Residual: 19.3956
N: 329

TEST SET PERFORMANCE
RMSE: 22.2100
MAE: 16.6972
R2: 0.4528
Correlation: 0.7009
Mean_Residual: -4.4293
N: 283

RESULTS SUMMARY
          Metric  Validation       Test
           RMSE   40.825201  22.210010
            MAE   30.625273  16.697227
             R2    0.310723   0.452813
Correlation (R)    0.732872   0.700890
  Mean Residual   19.395573  -4.429279
              N  329.000000 283.000000

TOP 10 COMBINATIONS
  C  nu  mean_test_score   CV_rmse
128 0.4      -918.031321 30.299032
256 0.4      -942.463945 30.699576
512 0.2      -951.880760 30.852565
128 0.9      -951.999677 30.854492
128 0.6      -952.519933 30.862922
128 0.5      -953.259334 30.874898
128 0.1      -953.673065 30.881598
128 0.2      -954.829829 30.900321
128 0.3      -957.91007

In [17]:
best_params_knn = gs_knn.best_params_
best_params_svr = gs_svr.best_params_
best_params_nusvr = gs_nusvr.best_params_


with open('results.json', 'r') as f:
    results = json.load(f)

results["best_params"] = results.get("best_params", {})
results["best_params"]["best_params_publications_included"] = {
    "knn": best_params_knn,
    "svr": best_params_svr,
    "nusvr": best_params_nusvr,
}

with open('results.json', "w") as f:
    json.dump(results, f, indent=2)

print(results["best_params"])


{'best_params_publications_included': {'knn': {'model__metric': 'manhattan', 'model__n_neighbors': 12, 'model__p': 2, 'model__weights': 'distance'}, 'svr': {'model__C': 128, 'model__epsilon': 0.01}, 'nusvr': {'model__C': 128, 'model__nu': 0.4}}, 'recoded_50': {'knn': {'model__metric': 'minkowski', 'model__n_neighbors': 3, 'model__p': 1, 'model__weights': 'distance'}, 'svr': {'model__C': 1024, 'model__epsilon': 3, 'model__gamma': 'scale'}, 'nusvr': {'model__C': 1024, 'model__gamma': 'scale', 'model__nu': 0.4}}}
